# Credit Default Prediction — Part 1: Exploratory Data Analysis

**Business context:** Before building a model, we need to understand who is in our borrower population, what data quality issues exist, and which features show the strongest signal for repayment risk. This notebook answers those questions.

Findings here directly inform feature engineering decisions in notebook 02.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use('seaborn-v0_8-whitegrid')
pd.set_option('display.max_columns', None)

df = pd.read_csv('../data/cs-training.csv', index_col=0)
print(f'Dataset shape: {df.shape}')
df.head()

## 1. Target Variable: Class Imbalance

Consumer lending datasets are almost always imbalanced — default is a rare event. Understanding the base rate sets expectations for model performance and informs our evaluation metric choice.

In [ ]:
target = 'SeriousDlqin2yrs'
default_rate = df[target].mean()
print(f'Overall default rate: {default_rate:.2%}')
print(f'Non-default: {(1 - default_rate):.2%}')

fig, ax = plt.subplots(figsize=(6, 4))
df[target].value_counts().plot(kind='bar', ax=ax, color=['#2ecc71', '#e74c3c'], edgecolor='white')
ax.set_xticklabels(['No Default', 'Default'], rotation=0)
ax.set_title('Class Distribution: Loan Default', fontsize=13)
ax.set_ylabel('Count')
for p in ax.patches:
    ax.annotate(f'{p.get_height():,.0f}', (p.get_x() + p.get_width() / 2, p.get_height()),
                ha='center', va='bottom')
plt.tight_layout()
plt.savefig('../outputs/figures/class_distribution.png', dpi=150)
plt.show()

## 2. Missing Value Analysis

In [ ]:
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_summary = pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct})
missing_summary = missing_summary[missing_summary['Missing Count'] > 0].sort_values('Missing %', ascending=False)
print('Features with missing values:')
print(missing_summary)

# Note: MonthlyIncome (~20% missing) and NumberOfDependents (~3% missing)
# Imputation strategy will be defined in notebook 02

## 3. Feature Distributions by Default Status

For each key feature, we compare the distribution for borrowers who defaulted vs. those who did not. This reveals which features carry the most predictive signal.

In [ ]:
key_features = [
    'RevolvingUtilizationOfUnsecuredLines',
    'age',
    'DebtRatio',
    'MonthlyIncome',
    'NumberOfOpenCreditLinesAndLoans',
    'NumberOfTimes90DaysLate'
]

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()

for i, feat in enumerate(key_features):
    for label, color in zip([0, 1], ['#2ecc71', '#e74c3c']):
        subset = df[df[target] == label][feat].dropna()
        # Cap extreme outliers for visualization
        cap = subset.quantile(0.99)
        subset = subset[subset <= cap]
        axes[i].hist(subset, bins=40, alpha=0.5, color=color,
                     label='Default' if label == 1 else 'No Default', density=True)
    axes[i].set_title(feat, fontsize=10)
    axes[i].legend(fontsize=8)

plt.suptitle('Feature Distributions by Default Status', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig('../outputs/figures/feature_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Default Rate by Key Segments

Understanding how default rates vary across borrower segments is essential for both model development and fair lending compliance.

In [ ]:
# Default rate by age group
df['age_group'] = pd.cut(df['age'], bins=[0, 25, 35, 45, 55, 65, 100],
                          labels=['<25', '25-34', '35-44', '45-54', '55-64', '65+'])
age_default = df.groupby('age_group')[target].mean().reset_index()
age_default.columns = ['Age Group', 'Default Rate']

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.bar(age_default['Age Group'], age_default['Default Rate'],
               color='#3498db', edgecolor='white')
ax.axhline(default_rate, color='red', linestyle='--', label=f'Overall rate ({default_rate:.1%})')
ax.set_title('Default Rate by Age Group', fontsize=13)
ax.set_ylabel('Default Rate')
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{y:.0%}'))
ax.legend()
for bar, rate in zip(bars, age_default['Default Rate']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.002,
            f'{rate:.1%}', ha='center', va='bottom', fontsize=9)
plt.tight_layout()
plt.savefig('../outputs/figures/default_rate_by_age.png', dpi=150)
plt.show()

print('\nFair lending note: Significant variation in default rates by age group')
print('This will be revisited in notebook 04 to check for model disparate impact.')

## 5. Correlation Analysis

In [ ]:
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
numeric_cols = [c for c in numeric_cols if c != target]

corr_with_target = df[numeric_cols].corrwith(df[target]).sort_values(ascending=False)
print('Feature correlations with default (SeriousDlqin2yrs):')
print(corr_with_target.round(3))

fig, ax = plt.subplots(figsize=(8, 5))
colors = ['#e74c3c' if v > 0 else '#2ecc71' for v in corr_with_target]
corr_with_target.plot(kind='barh', ax=ax, color=colors, edgecolor='white')
ax.axvline(0, color='black', linewidth=0.8)
ax.set_title('Feature Correlation with Default', fontsize=13)
ax.set_xlabel('Pearson Correlation')
plt.tight_layout()
plt.savefig('../outputs/figures/feature_correlation.png', dpi=150)
plt.show()

## EDA Summary

| Finding | Implication |
|---|---|
| Default rate ~6.7% | Significant class imbalance; use AUC-ROC and precision-recall, not accuracy |
| MonthlyIncome has ~20% missing | Median imputation by age group; flag as binary indicator feature |
| RevolvingUtilization has extreme outliers (>1.0) | Cap at 1.0 — values >1 are data errors |
| Late payment history (90d+) strongest correlate | Strong behavioral signal; engineer cumulative delinquency feature |
| Younger borrowers (<25) have higher default rates | Flag for fair lending review in model output |

➡️ Proceed to `02_feature_engineering.ipynb`